In [16]:
"""
split_gel_data.py
=================
Reads an equilibrated LAMMPS data file (atom_style molecular, 5 atom types)
and writes two new data files:

  <stem>_polymer_only.data  — types 1 (crosslink) and 2 (chain bead) only
                               positions unchanged; bonds preserved
  <stem>_solvent_only.data  — types 3 (solvent), 4 (support), 5 (piston) only
                               positions unchanged; no bonds
                               type remapping: 3→1, 4→2, 5→3

Usage
-----
  python split_gel_data.py <input.data> [output_stem]

  If output_stem is omitted the input filename (minus .data) is used.

Atom types in the original file
--------------------------------
  1  Crosslink  (polymer)
  2  Chain bead (polymer)
  3  Solvent
  4  Bottom support (frozen)
  5  Top piston  (mobile)
"""

import sys
import os
import re


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def parse_data_file(path):
    """
    Parse a LAMMPS data file (atom_style molecular).
    Returns a dict with keys:
        header    : dict of scalar counts/bounds
        masses    : list of (type_id, mass, comment)
        atoms     : list of dicts {id, mol, type, x, y, z}
        bonds     : list of dicts {id, type, atom1, atom2}
    """
    with open(path) as f:
        lines = f.readlines()

    header = {
        'n_atoms': 0, 'n_bonds': 0,
        'n_angles': 0, 'n_dihedrals': 0, 'n_impropers': 0,
        'n_atom_types': 0, 'n_bond_types': 0,
        'xlo': 0.0, 'xhi': 0.0,
        'ylo': 0.0, 'yhi': 0.0,
        'zlo': 0.0, 'zhi': 0.0,
        'title': ''
    }
    masses = []
    atoms  = []
    bonds  = []

    section = None
    for i, raw in enumerate(lines):
        line = raw.strip()

        # Title is the very first non-blank line
        if i == 0:
            header['title'] = line
            continue

        if not line or line.startswith('#'):
            continue

        # Strip inline comments (e.g. "Atoms # molecular", "Pair Coeffs # lj/cut/omp")
        line = line.split('#')[0].strip()
        if not line:
            continue

        # ---- Section headers (letters and spaces only, no digits) ----
        if re.match(r'^[A-Za-z][A-Za-z /]+$', line):
            section = line
            continue

        # ---- Header keyword lines ----
        if section is None:
            for kw, key in [
                ('atoms',        'n_atoms'),
                ('bonds',        'n_bonds'),
                ('angles',       'n_angles'),
                ('dihedrals',    'n_dihedrals'),
                ('impropers',    'n_impropers'),
                ('atom types',   'n_atom_types'),
                ('bond types',   'n_bond_types'),
            ]:
                if line.endswith(kw):
                    header[key] = int(line.split()[0])
                    break

            m = re.match(r'^([-\d.eE+]+)\s+([-\d.eE+]+)\s+xlo\s+xhi', line)
            if m:
                header['xlo'], header['xhi'] = float(m.group(1)), float(m.group(2))
            m = re.match(r'^([-\d.eE+]+)\s+([-\d.eE+]+)\s+ylo\s+yhi', line)
            if m:
                header['ylo'], header['yhi'] = float(m.group(1)), float(m.group(2))
            m = re.match(r'^([-\d.eE+]+)\s+([-\d.eE+]+)\s+zlo\s+zhi', line)
            if m:
                header['zlo'], header['zhi'] = float(m.group(1)), float(m.group(2))
            continue

        # ---- Section body ----
        if section == 'Masses':
            # id  mass  [# comment]
            parts = line.split('#', 1)
            nums  = parts[0].split()
            comment = parts[1].strip() if len(parts) > 1 else ''
            masses.append((int(nums[0]), float(nums[1]), comment))

        elif section == 'Atoms':
            # id mol type x y z  [image flags ignored]
            parts = line.split()
            atoms.append({
                'id':  int(parts[0]),
                'mol': int(parts[1]),
                'type': int(parts[2]),
                'x': float(parts[3]),
                'y': float(parts[4]),
                'z': float(parts[5]),
            })

        elif section == 'Bonds':
            parts = line.split()
            bonds.append({
                'id':    int(parts[0]),
                'type':  int(parts[1]),
                'atom1': int(parts[2]),
                'atom2': int(parts[3]),
            })

    return {'header': header, 'masses': masses, 'atoms': atoms, 'bonds': bonds}


def write_data_file(path, title, box, atom_types_info, atoms, bonds=None):
    """
    Write a LAMMPS data file (atom_style molecular).

    Parameters
    ----------
    path           : output path
    title          : header comment string
    box            : dict with xlo/xhi/ylo/yhi/zlo/zhi
    atom_types_info: list of (new_type_id, mass, label_comment)
    atoms          : list of dicts {id, mol, type, x, y, z}
    bonds          : list of dicts {id, type, atom1, atom2}  (None → no bonds)
    """
    n_bond_types = len(set(b['type'] for b in bonds)) if bonds else 0

    with open(path, 'w') as f:
        f.write(f"{title}\n\n")
        f.write(f"{len(atoms)} atoms\n")
        f.write(f"{len(bonds) if bonds else 0} bonds\n")
        f.write("0 angles\n")
        f.write("0 dihedrals\n")
        f.write("0 impropers\n\n")
        f.write(f"{len(atom_types_info)} atom types\n")
        if n_bond_types:
            f.write(f"{n_bond_types} bond types\n")
        f.write("\n")
        f.write(f"{box['xlo']:.6f} {box['xhi']:.6f} xlo xhi\n")
        f.write(f"{box['ylo']:.6f} {box['yhi']:.6f} ylo yhi\n")
        f.write(f"{box['zlo']:.6f} {box['zhi']:.6f} zlo zhi\n")
        f.write("\nMasses\n\n")
        for tid, mass, label in atom_types_info:
            comment = f"  # {label}" if label else ''
            f.write(f"{tid} {mass:.1f}{comment}\n")

        f.write("\nAtoms\n\n")
        for a in atoms:
            f.write(f"{a['id']} {a['mol']} {a['type']} "
                    f"{a['x']:.6f} {a['y']:.6f} {a['z']:.6f}\n")

        if bonds:
            f.write("\nBonds\n\n")
            for b in bonds:
                f.write(f"{b['id']} {b['type']} {b['atom1']} {b['atom2']}\n")


# ---------------------------------------------------------------------------
# Main splitting logic
# ---------------------------------------------------------------------------

def split_gel(input_path, output_stem=None):
    print(f"Reading: {input_path}")
    data = parse_data_file(input_path)

    header = data['header']
    atoms  = data['atoms']
    bonds  = data['bonds']

    # Build a lookup: original_atom_id → atom dict
    atom_by_id = {a['id']: a for a in atoms}

    box = {k: header[k] for k in ('xlo', 'xhi', 'ylo', 'yhi', 'zlo', 'zhi')}

    if output_stem is None:
        output_stem = os.path.splitext(input_path)[0]

    # -----------------------------------------------------------------------
    # Build original masses dict: type → (mass, comment)
    # -----------------------------------------------------------------------
    orig_masses = {t: (m, c) for t, m, c in data['masses']}

    # -----------------------------------------------------------------------
    # 1.  POLYMER-ONLY  (types 1 and 2 kept, no remapping needed)
    # -----------------------------------------------------------------------
    POLYMER_TYPES = {1, 2}

    poly_atoms_raw  = [a for a in atoms if a['type'] in POLYMER_TYPES]
    poly_ids_old    = {a['id'] for a in poly_atoms_raw}

    # Only keep bonds where BOTH endpoints are polymer
    poly_bonds_raw  = [b for b in bonds
                       if b['atom1'] in poly_ids_old and b['atom2'] in poly_ids_old]

    # Renumber atoms 1..N
    poly_id_map = {a['id']: new_id for new_id, a in enumerate(poly_atoms_raw, start=1)}

    poly_atoms = []
    for a in poly_atoms_raw:
        poly_atoms.append({
            'id':   poly_id_map[a['id']],
            'mol':  a['mol'],
            'type': a['type'],          # 1 and 2 stay as 1 and 2
            'x': a['x'], 'y': a['y'], 'z': a['z'],
        })

    poly_bonds = []
    for new_bid, b in enumerate(poly_bonds_raw, start=1):
        poly_bonds.append({
            'id':    new_bid,
            'type':  b['type'],
            'atom1': poly_id_map[b['atom1']],
            'atom2': poly_id_map[b['atom2']],
        })

    poly_type_info = []
    for t in sorted(POLYMER_TYPES):
        mass, label = orig_masses.get(t, (1.0, ''))
        poly_type_info.append((t, mass, label))

    poly_out = f"{output_stem}_polymer_only.data"
    write_data_file(
        poly_out,
        title="LAMMPS data file — polymer only (crosslinks + chain beads)",
        box=box,
        atom_types_info=poly_type_info,
        atoms=poly_atoms,
        bonds=poly_bonds,
    )

    print(f"\nPolymer-only file: {poly_out}")
    print(f"  Atom types kept : 1 (crosslink), 2 (chain bead)  [no remapping]")
    print(f"  Atoms           : {len(poly_atoms)}")
    print(f"  Bonds           : {len(poly_bonds)}")

    # -----------------------------------------------------------------------
    # 2.  SOLVENT-ONLY  (types 3, 4, 5 kept; remap 3→1, 4→2, 5→3)
    # -----------------------------------------------------------------------
    SOLVENT_TYPES   = {3, 4, 5}
    TYPE_REMAP      = {3: 1, 4: 2, 5: 3}   # old → new
    TYPE_LABELS_NEW = {
        1: 'Solvent (was type 3)',
        2: 'Bottom support — frozen (was type 4)',
        3: 'Top piston — mobile (was type 5)',
    }

    solv_atoms_raw = [a for a in atoms if a['type'] in SOLVENT_TYPES]

    # Renumber atoms 1..N
    solv_id_map = {a['id']: new_id for new_id, a in enumerate(solv_atoms_raw, start=1)}

    # Find which original types are actually present
    present_orig_types = sorted({a['type'] for a in solv_atoms_raw})
    present_new_types  = sorted({TYPE_REMAP[t] for t in present_orig_types})

    solv_atoms = []
    for a in solv_atoms_raw:
        solv_atoms.append({
            'id':   solv_id_map[a['id']],
            'mol':  a['mol'],
            'type': TYPE_REMAP[a['type']],
            'x': a['x'], 'y': a['y'], 'z': a['z'],
        })

    solv_type_info = []
    for new_t in present_new_types:
        old_t = {v: k for k, v in TYPE_REMAP.items()}[new_t]
        mass, _ = orig_masses.get(old_t, (1.0, ''))
        solv_type_info.append((new_t, mass, TYPE_LABELS_NEW[new_t]))

    solv_out = f"{output_stem}_solvent_only.data"
    write_data_file(
        solv_out,
        title="LAMMPS data file — solvent only (+ support/piston if present)",
        box=box,
        atom_types_info=solv_type_info,
        atoms=solv_atoms,
        bonds=None,     # no bonds in solvent-only file
    )

    print(f"\nSolvent-only file: {solv_out}")
    print(f"  Type remapping  : 3→1 (solvent), 4→2 (support), 5→3 (piston)")
    print(f"  New types present: {present_new_types}")
    print(f"  Atoms           : {len(solv_atoms)}")
    per_new = {nt: sum(1 for a in solv_atoms if a['type'] == nt) for nt in present_new_types}
    for nt, cnt in per_new.items():
        print(f"    type {nt}: {cnt} atoms")

    # -----------------------------------------------------------------------
    # Summary
    # -----------------------------------------------------------------------
    n_orig = len(atoms)
    n_poly = len(poly_atoms)
    n_solv = len(solv_atoms)
    print(f"\nSanity check: {n_poly} + {n_solv} = {n_poly + n_solv}  "
          f"(original: {n_orig})")
    if n_poly + n_solv != n_orig:
        print("  WARNING: atom counts don't add up — check for unexpected type IDs.")
    else:
        print("  OK — all atoms accounted for.")


In [17]:
# ========== Inputs ==========

input_path  = "../../lammps_data_files_local/final_config_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000.data"
output_stem = None

split_gel(input_path, output_stem)


Reading: ../../lammps_data_files_local/final_config_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000.data

Polymer-only file: ../../lammps_data_files_local/final_config_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000_polymer_only.data
  Atom types kept : 1 (crosslink), 2 (chain bead)  [no remapping]
  Atoms           : 96214
  Bonds           : 108864

Solvent-only file: ../../lammps_data_files_local/final_config_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000_solvent_only.data
  Type remapping  : 3→1 (solvent), 4→2 (support), 5→3 (piston)
  New types present: [1, 2, 3]
  Atoms           : 305783
    type 1: 197459 atoms
    type 2: 54162 atoms
    type 3: 54162 atoms

Sanity check: 96214 + 305783 = 401997  (original: 401997)
  OK — all atoms accounted for.
